# DRVI on the malignant epithelial subset

Phase 04 ran DRVI on the **epithelial** compartment and said in its own README what it could
not claim: `cell_type` there comes from a CellTypist model trained on the *normal* adult
breast, that model has no malignant class, and so its 74,441 cells are a mixture of normal
and tumour epithelium that the labels cannot separate. 05_1 called CNV per cell and 05_2
rebuilt the whole pre-processing on the aneuploid cells alone. This notebook is 04_2's
question asked of that object: what the DRVI latent dimensions say about **36,192 malignant
cells** from 19 cohorts.

Same recipe as `04_2_drvi_run/drvi_epi.ipynb` - same architecture, same `batch_key='cohort'`,
same 400 full epochs, same seed - with three things changed:

- the input is `shiao_tum_hvg_2k.h5ad` (36,192 cells x 2,000 HVGs re-selected **inside the
  tumour** by 05_2), not the epithelial `shiao_epi_hvg_2k.h5ad`;
- **`N_LATENT = 32`**, not 64. 04 settled at 64 *after* 32 left none of its dimensions
  vanished on twice these cells; here the scan starts at 32 again, and the vanished count
  below is what decides whether it stays;
- **`cell_type` is constant.** Every cell in this object is `malignant`, so it groups
  nothing. Everything that 04_2 draws by cell type is drawn here by `optscib_tum_leiden`,
  the clustering 05_2 computed on these very cells, and - second, as a landmark -
  by `cell_type_01_4`, the pre-CNV CellTypist label.

**None of these labels is an input.** `DRVI.setup_anndata` takes a `labels_key` and it is left
`None` below: the counts layer and `batch_key='cohort'` are the entire input, so the latent
space is built without knowing any of them and none of them can bias it. They are read back
afterwards, as captions on dimensions that already exist - which is what makes an imperfect
grouping affordable: a wrong caption can mislead a reader, it cannot corrupt the model.
Dropping it would remove the caption, not a source of error, and leave 32 anonymous axes.

Nothing is inherited from 04 - not the model, not the HVGs, not the latent size. As in 02_2,
03_2 and 04_2 the size is checked by eye, from how many dimensions vanish, so this notebook
is run once per candidate size. Everything that depends on that choice (model, embedding,
figures, tables) is derived from `N_LATENT` in the configuration cell below, so a re-run at
64 never overwrites the 32.

> **What `cell_type_01_4` is, and is not.** It is what CellTypist called these cells *before*
> the CNV call, with a model that has no malignant class - so as an **identity it is obsolete,
> and establishing that is what phase 05 is for.** It is kept as a grouping because on this
> subset it is the only non-constant CellTypist column left: 05_1's post-CNV re-annotation ran
> on the non-malignant cells and leaves `celltypist_predicted_cnv` at the constant `malignant`
> here, while the raw pre-voting `celltypist_predicted` puts Fibro-matrix on 1,871 aneuploid
> cells and pericytes on 658. The same substitution, with the same argument, is what
> `05_2/clustering_tum.py` does for its NMI target.
>
> **Neither grouping is clean, and they fail differently.** The CellTypist label comes from the
> wrong model. The leiden partition comes from the *unintegrated* PCA of 05_2, so it carries part
> of the cohort structure DRVI is meant to correct - which is why the heatmaps below are drawn
> against `cohort` as well - and it is not fully independent of the label above either:
> `clustering_tum.py` picked its **resolution** by maximising NMI against `cell_type_01_4`
> (recorded in `uns['optscib_tum_leiden_label_key']`). The clusters come from the expression
> graph; the borrowed label chose how many of them there are, not which cell goes in which.
> A dimension that answers to both and not to `cohort` is the safe reading; one that answers to
> leiden alone is a state the normal-breast vocabulary has no word for, which is what this phase
> is looking for.
>
> **The caption that owes nothing to any annotation** is the third one, and this notebook writes
> it too: the OOD / IND interpretability scores, which name a dimension by its genes. That is the
> route 05_6 and 05_7 take; the two groupings here are the cross-check on it, never the evidence.

## Libraries

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import drvi
from drvi.model import DRVI

import warnings
warnings.filterwarnings("ignore")

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Scanpy setting: clean white background and high resolution figures
sc.settings.set_figure_params(dpi=300, facecolor="white")
# Set the level of verbosity of scanpy to get more detailed info
sc.settings.verbosity = 3

# Tell Jupyter to show all outputs and not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

!pip list

## Configuration

The only cell to edit when changing the latent size: the run id, the model, the embedding,
the figure folder and the tables all follow from `N_LATENT`.

The paths come from `05_2_subsetting/cell_set.py` rather than being spelled out, so this
notebook follows `CELL_SET` like every script of the phase: unset (or `tum`) is the malignant
subset, `CELL_SET=epi` the control set of all epithelium under the post-CNV labels. The two
produce different run ids - `drvi_tum_32` and `drvi_epicnv_32` - and therefore different
models, embeddings, figures and tables.

In [ ]:
N_LATENT = 32        # latent dimensions, the run of this phase (re-run at 64 to compare what vanishes)
SEED = 123
N_EPOCHS = 400       # the full run: scvi-tools warms the KL term up over exactly these
EARLY_STOPPING = False   # off by default, see the training section
OVERWRITE = False    # False reuses the model / embedding already on disk

PROJECT_DIR = Path("/home/albertoc/Desktop/QCB-Master-Thesis")
PHASE_DIR = PROJECT_DIR / "05_drvi_tumoral_epi"

# The CELL_SET -> prefix mapping and every threshold of this phase live in one module, next
# to the scripts that wrote the input. `run_drvi_tum.py` imports the same one.
os.environ.setdefault("DATA_DIR", str(PROJECT_DIR / "datasets"))
sys.path.insert(0, str(PHASE_DIR / "05_2_subsetting"))
import cell_set as C

C.banner("05_3 DRVI run")

BATCH_KEY = C.BATCH_KEY           # 'cohort', as in phase 02, 03, 04 and in 05_2
LABEL_KEY = C.LABEL_KEY           # 'cell_type': the post-CNV label, CONSTANT under CELL_SET=tum
GROUP_KEY = C.PRIOR_LABEL_KEY     # 'cell_type_01_4': the pre-CNV label, what actually groups
LEIDEN_KEY = "optscib_tum_leiden" # 05_2's clustering, as named by clustering_tum.py

# The compartment goes into the run id, so nothing here can be confused with 04's
# `drvi_epi_*`, 03_2's `drvi_nonimm_*` or 02_2's whole-dataset `drvi_unscaled_*`, and the
# latent size keeps the sizes side by side.
RUN_ID = f"drvi_{C.compartment()}_{N_LATENT}"

DATA_DIR = C.data_dir()
TUM_DIR = C.tum_dir()             # $DATA_DIR/05_tum, where 05_1 and 05_2 already wrote

# Input: the 2,000-gene object written by 05_2's reduce_data_tum.py. HVGs were selected on
# the malignant cells only, .layers['counts'] are the raw counts DRVI trains on.
INPUT_H5AD = C.path("_hvg_2k.h5ad")
# The definitive object of 05_2 (all genes, scran log-norm, leiden): read for the leiden
# column, and at the end to carry the latent space over for the steps after.
FULL_H5AD = C.path(".h5ad")

# Outputs, all under DATA_DIR and never in the repo (the embedding alone is a few hundred
# MB, past GitHub's per-file limit, and /datasets/* is gitignored).
MODEL_PATH = TUM_DIR / f"model_{RUN_ID}.pt"       # one flat file per run (see save_model)
LATENT_H5AD = TUM_DIR / f"embed_{RUN_ID}.h5ad"    # latent space + interpretability scores
DOWNSTREAM_H5AD = C.path(f"_{RUN_ID}.h5ad")       # the 05_2 object + obsm['X_drvi']

print(f"run id      {RUN_ID}")
print(f"input       {INPUT_H5AD}")
print(f"model       {MODEL_PATH}")
print(f"latent      {LATENT_H5AD}")
print(f"downstream  {DOWNSTREAM_H5AD}")

### Figures and tables

Every figure of this notebook goes to `05_drvi_tumoral_epi/figures/05_3_<run_id>/`, next to
the `05_1_*` and `05_2_*` folders of the steps before, and every table it writes to
`05_drvi_tumoral_epi/tables/05_3_<run_id>/`. The split is the point: `figures/` holds images
and nothing else.

In [ ]:
FIG_DIR = PHASE_DIR / "figures" / f"05_3_{RUN_ID}"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = FIG_DIR

TABLE_DIR = PHASE_DIR / "tables" / f"05_3_{RUN_ID}"
TABLE_DIR.mkdir(parents=True, exist_ok=True)


def savefig(name, fig=None, dpi=300):
    """Save a matplotlib figure (the current one by default) into FIG_DIR.

    For the DRVI and seaborn plots, which draw on the pyplot state instead of going through
    sc.pl and so ignore sc.settings.figdir. The run id is appended to the name, so figures
    from different latent sizes stay distinguishable even once they are pulled out of their
    folder.
    """
    fig = plt.gcf() if fig is None else fig
    path = FIG_DIR / f"{name}_{RUN_ID}.png"
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"[fig] {path}")
    return path


print(f"figures     {FIG_DIR}")
print(f"tables      {TABLE_DIR}")

In [ ]:
print("Last run with scvi-tools version:", scvi.__version__)
print("Last run with DRVI version:", drvi.__version__)

### Model I/O

`scvi-tools` saves a model as a *directory* holding a file called `model.pt`, which would
mean one folder per run and the latent size buried in the folder name. These two helpers keep
that out of `05_tum/`: one flat `model_drvi_tum_<N>.pt` per run, next to its embedding. Same
pair as in 04_2 and in `run_drvi_tum.py`, so a model trained by any of the three loads back
in the other two.

In [ ]:
def save_model(model, path):
    """Save a DRVI model as the single file `path`.

    model.save() writes `<dir>/model.pt` and gives no say over the file name, so it writes
    into a scratch directory *inside the destination folder* (same filesystem, so the move
    below is a rename and not a copy) and the one file it produced is then renamed.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=path.parent) as scratch:
        model.save(scratch, overwrite=True)
        Path(scratch, "model.pt").replace(path)
    return path


def load_model(path, adata):
    """Load a DRVI model from the single file `path`; the counterpart of save_model.

    DRVI.load() insists on a directory containing `model.pt`, so it gets a scratch one
    holding a symlink to the real file: nothing is copied, and the scratch is gone by the
    time this returns.
    """
    path = Path(path).resolve()
    with tempfile.TemporaryDirectory(dir=path.parent) as scratch:
        os.symlink(path, Path(scratch, "model.pt"))
        return DRVI.load(scratch, adata)

## Load dataset

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)
adata

print(f"{adata.n_obs:,} cells x {adata.n_vars:,} HVGs")
print("First 10 HVG genes:", list(adata.var_names[:10]))

# Sanity checks on what 05_2 produced: the right cell set and the counts layer present.
# `compartment` is what tells this object from 04's ('epi') and from 03's ('non_imm').
assert adata.obs[C.COMPARTMENT_KEY].astype(str).nunique() == 1, "not a single-compartment object"
assert adata.obs[C.COMPARTMENT_KEY].astype(str).iloc[0] == C.compartment(), \
    f"compartment is not {C.compartment()!r}: this is not the CELL_SET={C.cell_set()} object"
assert "counts" in adata.layers, "raw counts layer missing, DRVI has nothing to train on"
# No cell without a CNV call can be here: under `tum` by construction, under `epi` because
# 05_2 drops them rather than assuming they are normal.
assert (adata.obs[C.STATUS_KEY].astype(str) != "not_tested").all(), \
    "not_tested cells in the input: 05_2 should have dropped them"

print("compartment :", adata.obs[C.COMPARTMENT_KEY].astype(str).unique().tolist())
print("cnv_status  :", sorted(adata.obs[C.STATUS_KEY].astype(str).unique().tolist()))
print("cell_type   :", sorted(adata.obs[LABEL_KEY].astype(str).unique().tolist()))
print(f"{adata.obs[BATCH_KEY].nunique()} cohorts, {adata.obs[GROUP_KEY].nunique()} pre-CNV labels")

adata.obs[GROUP_KEY].value_counts()

**The label column is constant, and that is the object working as intended.** `cell_type` is
`malignant` for all 36,192 cells: it was written by 05_2 from the CNV call, not by CellTypist,
and it is what makes this phase different from 04. What it costs is every figure that groups
cells by label, and the substitute is `cell_type_01_4` - the label CellTypist gave these cells
before the call - which is not constant inside the tumour.

`cell_type_01_4` is used the way `05_2/clustering_tum.py` uses it for the NMI target: as **one
non-constant grouping**, to pick a number and to arrange a figure. Two of its levels
(`Lumsec-prol` and `Lumsec-basal`) are 84% of the subset, and the smallest of them has two
cells. Nothing downstream of this notebook treats it as biology.

## Setup the model
- Seed fixed for reproducibility
- `cohort` as the batch to correct, as in phase 02, 03, 04 and in the 05_2 reduction
- `n_latent` comes from the configuration cell (32 here)
- **no `labels_key`**: the parameter exists and is deliberately left unset, so no annotation
  enters the training and none of the groupings used further down can bias the latent space
- Encoder / decoder dims [256, 128] / [128, 256], against the [128, 128] defaults
- `dispersion='gene-batch'`: really important parameter for this TNBC dataset, and the
  reason 05_2 kept `MIN_CELLS_PER_COHORT = 200` from 04_1 - a dispersion column per gene
  *and* per batch cannot be fitted on a cohort of 14 malignant cells

In [ ]:
# Setup the anndata for scvi-tools
DRVI.setup_anndata(
    adata,
    layer="counts",       # raw unnormalised data (used by default)
    batch_key=BATCH_KEY,  # the batch to fix
    # labels_key is left None on purpose: DRVI is unsupervised here, and `cell_type_01_4` /
    # `optscib_tum_leiden` are read against the latent space afterwards, never into it.
)

# Setting seed
scvi.settings.seed = SEED

# Setup the model
model = DRVI(
    adata,
    n_latent=N_LATENT,
    encoder_dims=[256, 128],
    decoder_dims=[128, 256],
    dispersion="gene-batch",   # really important parameter for TNBC
)

model

## Train the model
- `N_EPOCHS` (400) **full** epochs, as in the DRVI tutorial
- **No early stopping** (`EARLY_STOPPING = False`). scvi-tools raises the weight of the KL
  term linearly over `n_epochs_kl_warmup` epochs, 400 by default, so a run that stops earlier
  never trains at full KL weight - and it is that KL pressure that makes the unused dimensions
  vanish, the statistic `N_LATENT` is chosen on below. Turning it back on only makes sense
  together with a shorter warmup, e.g. `plan_kwargs={"n_epochs_kl_warmup": 150}`.
- Train / validation split is the scvi-tools default, 0.9 / 0.1

The lightest DRVI run of the thesis: 36,192 cells against the 74,441 of 04_2, the 176,610 of
03_2 and the 619,693 of 02_2, same architecture and half of 04_2's latent size. On the cluster
the identical run is `submit_drvi_tum.slurm`, which calls `run_drvi_tum.py` with the same
defaults; the model it writes loads back into the cell below.

In [ ]:
# Train only if the model is not on disk yet, or if OVERWRITE asks for a retrain.
# The training uses the GPU if one is available, otherwise the CPU.
if OVERWRITE or not MODEL_PATH.exists():
    model.train(
        max_epochs=N_EPOCHS,
        early_stopping=EARLY_STOPPING,   # False: the KL warmup needs all N_EPOCHS
        early_stopping_patience=50,      # only read when EARLY_STOPPING is True
    )
    save_model(model, MODEL_PATH)
    print(f"[model] saved to {MODEL_PATH}")
else:
    print(f"[model] {MODEL_PATH} already exists, not retraining (OVERWRITE=False)")

## Latent space + Graph + UMAP + PCA on latent space

In [ ]:
# Load the model back, so the cells below can be re-run without retraining
model = load_model(MODEL_PATH, adata)
model

In [ ]:
# The DRVI-specific view of the run: the latent space as .X, one var per latent dimension
# carrying its stats and interpretability scores.
if OVERWRITE or not LATENT_H5AD.exists():
    embed = ad.AnnData(model.get_latent_representation(), obs=adata.obs)

    # We set latent dimension statistics
    print("Setting latent dimension stats ...")
    model.set_latent_dimension_stats(embed, vanished_threshold=0.5)

    # We immediately calculate the interpretability gene scores with different approaches
    print("Calculating gene scores per factor ...")
    # out-of-distribution (OOD) approach uses decoder reconstructions to calculate gene scores (faster)
    model.calculate_interpretability_scores(embed, "OOD")
    # within-distribution (IND) approach iterates over all cells and calculates gene scores
    model.calculate_interpretability_scores(embed, "IND")

    print("Dimension reduction ...")
    sc.pp.neighbors(embed, n_neighbors=15, use_rep="X", n_pcs=embed.X.shape[1])
    sc.tl.umap(embed)
    sc.pp.pca(embed)

    print(f"[write] {LATENT_H5AD}")
    embed.write_h5ad(LATENT_H5AD)
else:
    print(f"[read] {LATENT_H5AD}")
    embed = sc.read_h5ad(LATENT_H5AD)

embed

### The 05_2 clustering, attached

`optscib_tum_leiden` is the only grouping of these cells that was computed *on these cells*:
it comes out of `05_2/clustering_tum.py`, which swept the resolution on the unintegrated PCA
of the subset. It is not in `INPUT_H5AD` - `reduce_data_tum.py` wrote that file before the
clustering ran - so it is fetched from the definitive object, by cell name.

Two things are then available to read the latent space against: a **borrowed annotation**
(`cell_type_01_4`, from a normal-breast model) and an **internal partition** (leiden, from these
cells' own expression). They are not fully independent - the leiden *resolution* was chosen by
maximising NMI against that same borrowed label, though the clusters themselves come from the
expression graph - but they are far from the same thing, and where a factor lines up with both
the reading is much safer than where it lines up with the CellTypist label alone.

Only `obs` is read out of the h5ad, a few MB out of ~330.

In [ ]:
try:
    from anndata.io import read_elem
except ImportError:              # anndata < 0.11
    from anndata.experimental import read_elem
import h5py

if LEIDEN_KEY not in embed.obs and FULL_H5AD.exists():
    with h5py.File(FULL_H5AD, "r") as f:
        _full_obs = read_elem(f["obs"])
    # By name, never by position: a reordering upstream must not pair a cell with another
    # cell's cluster.
    assert embed.obs_names.isin(_full_obs.index).all(), \
        "cells of the embedding are missing from the 05_2 object"
    embed.obs[LEIDEN_KEY] = _full_obs.loc[embed.obs_names, LEIDEN_KEY].values
    del _full_obs

HAS_LEIDEN = LEIDEN_KEY in embed.obs
print(f"{LEIDEN_KEY}: {embed.obs[LEIDEN_KEY].nunique() if HAS_LEIDEN else 'not available'}"
      f"{' clusters' if HAS_LEIDEN else ''}")

# The categorical keys the latent dimensions are read against below. `cell_type`,
# `compartment`, `fraction` and `cnv_status` are all constant on this subset and are dropped
# with a line saying so, rather than drawn as a one-row heatmap.
# leiden first: it is the only grouping computed on these cells, and the first key is the one
# that leads the figures below. If it could not be attached, the CellTypist label takes that
# slot by default rather than by choice.
CANDIDATE_KEYS = [LEIDEN_KEY, GROUP_KEY, BATCH_KEY, "treatment", "response", "phase", LABEL_KEY]
GROUPING_KEYS = []
for key in CANDIDATE_KEYS:
    if key not in embed.obs:
        print(f"[skip] {key}: not an obs column")
    elif embed.obs[key].astype(str).nunique() < 2:
        print(f"[skip] {key}: constant ({embed.obs[key].astype(str).iloc[0]!r})")
    else:
        GROUPING_KEYS.append(key)

GROUPING_KEYS

## UMAPs of the DRVI latent space

`cell_type`, `compartment`, `fraction` and `cnv_status` are dropped from the panels: after the
05_2 subset all four are constant (`malignant`, `tum`, `non_imm`, `malignant`) and carry no
information. `dataset_origin` (the technical CD45 sort) is likewise left out, as everywhere in
Part 2.

Two panels have no counterpart in 04_2: **`cnv_score` and `cnv_corr`**, the two quantities
05_1 called the subset with. They are the check that this latent space is about tumour
*states* and not about how aneuploid a cell is - a gradient running along a dominant dimension
would say the model spent its capacity on CNV burden, and would have to be read into
everything downstream. Their counterparts in the unintegrated space are
`figures/05_2_reduce_data/umap_tum_cnv_score.png` and `umap_tum_cnv_corr.png`.

The 05_2 figures - `figures/05_2_reduce_data/umap_tum_*.png` - are the same cells in the
*unintegrated* PCA space with the same keys and the same palettes, so each panel below has its
direct counterpart there: same cells, only the space changes.

In [ ]:
UMAP_SEED = 0

_order = np.random.default_rng(UMAP_SEED).permutation(embed.n_obs)
embed_plot = ad.AnnData(
    obs=embed.obs.iloc[_order].copy(),
    obsm={"X_umap": embed.obsm["X_umap"][_order]},
    # Palettes carried over from the input object, so a label keeps the same colour here and
    # in the 05_2 UMAPs of the same cells.
    uns={k: v for k, v in adata.uns.items() if k.endswith("_colors")},
)
embed_plot

In [ ]:
UMAP_QC_KEYS = {
    "cell_type_01_4": GROUP_KEY,     # leiden is inserted before it below, when available
    "cohort": BATCH_KEY,
    "treatment": "treatment",
    "response": "response",
    "phase": "phase",
    "cnv_score": "cnv_score",
    "cnv_corr": "cnv_corr",
    "n_genes_by_counts": "n_genes_by_counts",
    "total_counts": "total_counts",
    "mito": "pct_counts_mt",
    "ribo": "pct_counts_ribo",
    "size_factors": "size_factors",
}
if HAS_LEIDEN:
    UMAP_QC_KEYS = {"leiden": LEIDEN_KEY, **UMAP_QC_KEYS}
# `cell_type` joins the panels only where it says something - i.e. under CELL_SET=epi. On
# the malignant subset it is the constant 'malignant' and is left out, as above.
if embed_plot.obs[LABEL_KEY].astype(str).nunique() > 1:
    UMAP_QC_KEYS = {"cell_type": LABEL_KEY, **UMAP_QC_KEYS}

for label, col in UMAP_QC_KEYS.items():
    sc.pl.umap(embed_plot, color=col, save=f"_{label}_{RUN_ID}.png")

In [ ]:
UMAP_COMBINED_KEYS = ([LEIDEN_KEY] if HAS_LEIDEN else []) +     [GROUP_KEY, BATCH_KEY, "treatment", "response", "phase"]
if embed_plot.obs[LABEL_KEY].astype(str).nunique() > 1:       # CELL_SET=epi
    UMAP_COMBINED_KEYS = [LABEL_KEY] + UMAP_COMBINED_KEYS

with plt.rc_context({"figure.figsize": (7, 7)}):
    sc.pl.umap(
        embed_plot,
        color=UMAP_COMBINED_KEYS,
        ncols=2,
        wspace=0.8,
        hspace=0.25,
        save=f"_combined_{RUN_ID}.png",
    )

### One panel per group

One panel per level of each grouping, each highlighting only its own cells, says where each
group actually sits in the DRVI space. Both groupings get a figure, and they answer different
questions.

**leiden** - where each state computed on *these* cells lands once the cohorts are corrected.
A leiden cluster that stays compact here is a state both spaces agree on; one that scatters was
a feature of the unintegrated PCA, cohort structure included.

**`cell_type_01_4`** - whether DRVI's tumour states line up with the normal-breast identity
these cells were *mistaken for*. Nine of the eleven epithelial labels survive into the subset
and two of them, `Lumsec-prol` (18,617 cells) and `Lumsec-basal` (11,771), are 84% of it. Two
labels sitting on top of each other is the expected outcome here, not a failure: the model that
produced them had no malignant class to put any of them in.

In [ ]:
# Same order as GROUPING_KEYS: the partition computed on these cells first, the borrowed
# label second.
PANEL_KEYS = ([LEIDEN_KEY] if HAS_LEIDEN else []) + [GROUP_KEY]

for key in PANEL_KEYS:
    groups = embed_plot.obs[key].astype("category")
    labels = [c for c in groups.cat.categories if (groups == c).any()]

    ncols = 4
    nrows = int(np.ceil(len(labels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.4))
    axes = np.atleast_1d(axes).flatten()

    for ax, label in zip(axes, labels):
        n = int((groups == label).sum())
        sc.pl.umap(embed_plot, color=key, groups=[label], title=f"{label} ({n:,})", ax=ax,
                   show=False, legend_loc="none", na_in_legend=False, size=4)
        ax.set_xlabel("")
        ax.set_ylabel("")
    for ax in axes[len(labels):]:
        ax.set_visible(False)

    plt.tight_layout()
    savefig(f"umap_per_group_{key}")
    plt.show()

## t-SNE of the DRVI latent space

The same cells and the same latent space as the UMAPs above, in the other embedding: t-SNE
keeps local neighbourhoods and makes no claim about the distance between clusters, so it is
read as a second opinion on the UMAP and not as a replacement. Three panels - the pre-CNV
label, the batch and the leiden partition: whether the groups still sit together once the
cohorts are corrected, and whether the cohorts are mixed.

Its counterpart is `figures/05_2_reduce_data/umap_tum_cohort.png` and the panels beside it:
the same cells and the same keys in the *unintegrated* PCA space.

Test only, as in 05_2: `X_tsne` is computed here and kept in memory, the cached
`embed_drvi_tum_<N>.h5ad` on disk is not rewritten and nothing downstream reads it.

In [ ]:
# t-SNE on the DRVI latent space itself (`embed.X`, all N_LATENT dimensions), the same
# representation `sc.pp.neighbors` and the UMAP were built on.
# ~36k cells: a couple of minutes (scanpy falls back to sklearn's Barnes-Hut t-SNE).
TSNE_SEED = 0

sc.tl.tsne(embed, use_rep="X", perplexity=30, random_state=TSNE_SEED)

# Same random draw order as the UMAP panels, for the same reason: otherwise the last cohort in
# the object is painted on top of every other one. Palettes carried over from the input
# object, so a label keeps the same colour here, in the UMAPs above and in 05_2.
_order_tsne = np.random.default_rng(TSNE_SEED).permutation(embed.n_obs)
embed_tsne_plot = ad.AnnData(
    obs=embed.obs.iloc[_order_tsne].copy(),
    obsm={"X_tsne": embed.obsm["X_tsne"][_order_tsne]},
    uns={k: v for k, v in adata.uns.items() if k.endswith("_colors")},
)

TSNE_KEYS = ([LEIDEN_KEY] if HAS_LEIDEN else []) + [GROUP_KEY, BATCH_KEY]

with plt.rc_context({"figure.figsize": (7, 7)}):
    sc.pl.tsne(
        embed_tsne_plot,
        color=TSNE_KEYS,
        ncols=2,
        wspace=0.8,
        hspace=0.25,
        save=f"_combined_{RUN_ID}.png",
    )

## Latent dimensions

How many of the `N_LATENT` dimensions DRVI actually used. A *vanished* dimension carries no
signal: it is the model saying it did not need that much room, and the count is the number
that justifies the choice of `N_LATENT`. If almost nothing vanishes, the latent space is too
small and worth re-running larger.

**This is the cell to read before deciding whether to re-run at another size.** 32 is the
starting size here for the reason 04_2 gives for 64 being its own: the subset is half of 04's
(36,192 cells against 74,441) and its annotation is a single constant value rather than ten
labels. Several vanished dimensions mean 32 was already generous; **none means 32 is too tight
and the run should be repeated at 64** and the two compared - which is exactly what happened
to 04_2's first attempt at this size, on twice these cells (0 / 32 vanished).

In [ ]:
n_vanished = int(embed.var["vanished"].sum())
print(f"{n_vanished} vanished / {embed.n_vars} latent dimensions "
      f"({embed.n_vars - n_vanished} effectively used)")
if n_vanished == 0:
    print(f"[note] nothing vanished at N_LATENT={N_LATENT}: the space is too tight to say "
          f"what DRVI did not need. Re-run this notebook with N_LATENT={N_LATENT * 2} - it "
          f"writes beside this run, it does not replace it.")

# The dimensions that reconstruct the most, i.e. where to start reading the model.
embed.var.sort_values("reconstruction_effect", ascending=False).head(10)

In [ ]:
# Per-dimension statistics: reconstruction effect, max value, mean, std.
drvi.utils.pl.plot_latent_dimension_stats(embed, ncols=2, show=False)
savefig("latent_dimension_stats")

# The same, vanished dimensions dropped: the informative half of the plot above.
drvi.utils.pl.plot_latent_dimension_stats(embed, ncols=2, remove_vanished=True, show=False)
savefig("latent_dimension_stats_rmVanished")

In [ ]:
# Every non-vanished dimension on the UMAP (remove_vanished defaults to True here).
drvi.utils.pl.plot_latent_dims_in_umap(embed, show=False)
savefig("latent_dims_in_umap")

### Which dimension responds to which group

One heatmap per grouping in `GROUPING_KEYS`, the constant columns already dropped, leiden
first. The first two are the ones to read together: `optscib_tum_leiden` says which dimensions
line up with a partition computed on the tumour itself, `cell_type_01_4` which ones line up
with the normal-breast label these cells carried before the CNV call. A dimension that answers
to a leiden cluster and to nothing else is a **tumour state**; one that answers to a CellTypist
label as well has a normal-epithelium counterpart to name it after.

`cohort` is the one to read defensively: this is the batch DRVI was told to correct, so a
dimension that responds to a single cohort is either residual batch or a tumour that really is
its own state - and with 19 patients those two are not separable here.

In [ ]:
for i, key in enumerate(GROUPING_KEYS):
    drvi.utils.pl.plot_latent_dims_in_heatmap(embed, key, title_col="title", show=False)
    savefig(f"latent_dims_in_heatmap_{key}")
    plt.show()

    # The first key also gets the grouped ordering: sorted by category as well as by
    # dimension, which makes it obvious when one group owns a whole dimension.
    if i == 0:
        drvi.utils.pl.plot_latent_dims_in_heatmap(embed, key, title_col="title",
                                                  sort_by_categorical=True, show=False)
        savefig(f"latent_dims_in_heatmap_{key}_sorted")
        plt.show()

### Interpretability (OOD scores)

The out-of-distribution scores come from the decoder reconstructions, so they are fast and
they favour the genes that are *specific* to a dimension.

In [ ]:
model.plot_interpretability_scores(embed, adata, show=False)
savefig("ood_interpretability_scores")
plt.show()

In [ ]:
# Genes (rows) appear in adata order and are not sorted.
ood_scores = model.get_interpretability_scores(embed, adata)
ood_scores.iloc[:10, :10]

The two halves of `OOD_combined`: the min_possible and max_possible log-fold-changes of each
dimension in the OOD setting. See the paper appendix for how they combine.

In [ ]:
ood_max = model.get_interpretability_scores(embed, adata, key="OOD_max_possible")
ood_min = model.get_interpretability_scores(embed, adata, key="OOD_min_possible")
ood_max.iloc[:10, :10]
ood_min.iloc[:10, :10]

In [ ]:
model.plot_interpretability_scores(embed, adata, key="OOD_max_possible", show=False)
savefig("ood_max_interpretability_scores")
plt.show()

model.plot_interpretability_scores(embed, adata, key="OOD_min_possible", show=False)
savefig("ood_min_interpretability_scores")
plt.show()

### Interpretability (IND scores)

This approach iterates over all the cells and averages the effect of each latent factor on
each gene; the scores are already stored in `embed`.

They reflect the broad mechanistic effect of a dimension. Genes are not filtered for
uniqueness, so a gene shared by several dimensions keeps a high score in all of them: the
complete picture of what a factor moves, as opposed to the OOD scores' specific one.

In [ ]:
model.plot_interpretability_scores(embed, adata, key="IND_linear_weighted_mean", show=False)
savefig("ind_linear_weighted_mean")
plt.show()

In [ ]:
# Genes (rows) appear in adata order and are not sorted.
ind_scores = model.get_interpretability_scores(embed, adata, key="IND_linear_weighted_mean")
ind_scores.iloc[:10, :10]

## The subset in the DRVI space

`embed` above is the model's own view of the run - one var per latent dimension, its stats and
its interpretability scores - and by construction carries no genes. This is the other half:
the definitive 05_2 object (all genes, scran log-norm, leiden) with the latent space added as
`obsm['X_drvi']`, for the downstream steps that need genes and latent coordinates in the same
object (05_5 cytotrace2, 05_6 cell-first, 05_9 cycle-confound). 05_4 works from the embedding
alone and does not open this file.

In [ ]:
if OVERWRITE or not DOWNSTREAM_H5AD.exists():
    print(f"[read] {FULL_H5AD}")
    full = sc.read_h5ad(FULL_H5AD)

    # Cell order is not assumed: the latent space is realigned on the 05_2 object's cells by
    # name, so a reordering anywhere upstream cannot pair a cell with another cell's
    # coordinates.
    assert set(embed.obs_names) == set(full.obs_names), \
        "the embedding and the 05_2 object disagree on cells"
    full.obsm["X_drvi"] = np.asarray(embed[full.obs_names].X, dtype=np.float32)
    assert np.isfinite(full.obsm["X_drvi"]).all()

    print(f"Writing {DOWNSTREAM_H5AD.name} (gzip, a few minutes) ...")
    full.write_h5ad(DOWNSTREAM_H5AD, compression="gzip")
    print(f"[write] {DOWNSTREAM_H5AD} "
          f"({full.n_obs:,} x {full.n_vars:,}, obsm['X_drvi'] {full.obsm['X_drvi'].shape}, "
          f"{DOWNSTREAM_H5AD.stat().st_size / 1024 ** 3:.2f} GB on disk)")
    del full
else:
    print(f"[have] {DOWNSTREAM_H5AD} (OVERWRITE=True to rewrite)")

## A closer look at the dimensions owned by a single group

The heatmaps above say *which* dimension responds to which group and `latent_dims_in_umap`
draws every surviving dimension at once, but neither answers the question one dimension at a
time: does a dimension that looks group-specific in the heatmap really light up that
population, and only that population? This section puts the two views side by side on the same
UMAP and with the same point order - the group on the left, the dimension on the right.

DRVI dimensions are directional: `DR 20-` is the *negative* half of dimension 20 and `DR 20+`
the positive one, the two halves are independent programs and in general only one of them
belongs to the group. The right panel is therefore coloured by the projection on the chosen
half (`-z` for a `-` dimension, the convention of
`plot_latent_dims_in_umap(directional=True)`), so a higher value always means *further along
the named direction*. The colour scale is the dimension's own range widened to at least
`[-1, 1]`, so a weak dimension is not stretched to look strong.

Which pairs get drawn is decided by the SMI screening below, following the DRVI *Identifying
cell types of DRVI factors* tutorial, exactly as in 02_2 and 04_2.

In [ ]:
# Both panels are drawn on `embed_plot`, the shuffled view of the latent space built in the
# UMAP section above: same point order and same palettes as every other UMAP here. Rebuilt
# below if the kernel jumped straight from the embedding to this section.
if "embed_plot" not in globals():
    _order = np.random.default_rng(globals().get("UMAP_SEED", 0)).permutation(embed.n_obs)
    embed_plot = ad.AnnData(
        obs=embed.obs.iloc[_order].copy(),
        obsm={"X_umap": embed.obsm["X_umap"][_order]},
        uns={k: v for k, v in adata.uns.items() if k.endswith("_colors")},
    )
    print(f"[rebuilt] embed_plot: {embed_plot.n_obs:,} cells")
else:
    print(f"embed_plot: {embed_plot.n_obs:,} cells")

In [ ]:
def dim_values(dim_title, direction):
    """The cells' projection on one half of a latent dimension, plus its colour range.

    `direction` is "+" or "-": for "-" the dimension is negated, so the returned values are
    large and positive exactly where the cells sit far along the negative half. The range
    follows DRVI: the dimension's own min / max, widened to at least [-1, 1].
    """
    hits = embed.var_names[embed.var["title"] == dim_title]
    if len(hits) != 1:
        raise KeyError(f"{dim_title!r} is not a dimension of this run")
    var_name = hits[0]

    sign = 1.0 if direction == "+" else -1.0
    values = pd.Series(np.asarray(embed[:, var_name].X).ravel(), index=embed.obs_names)

    lo, hi = float(embed.var.loc[var_name, "min"]), float(embed.var.loc[var_name, "max"])
    if sign < 0:
        lo, hi = -hi, -lo

    return sign * values, (min(lo, -1.0), max(hi, 1.0))


def plot_dim_vs_group(group, dim_title, direction, group_key=None, size=4, figsize=(11.5, 4.8)):
    """Two panels on the same UMAP: the group on the left, its dimension on the right.

    `group_key` is the obs column `group` is a level of - `cell_type_01_4` or the leiden
    partition, never the constant `cell_type`. Both panels use `embed_plot`, so the point
    order and the palettes are the ones of every other UMAP in this notebook. Returns the
    figure, for `savefig`.
    """
    group_key = GROUP_KEY if group_key is None else group_key
    label = f"{dim_title}{direction}"
    values, (vmin, vmax) = dim_values(dim_title, direction)
    values = values.loc[embed_plot.obs_names]

    # A light companion object: only the UMAP and the one column to colour by.
    panel = ad.AnnData(
        obs=pd.DataFrame({label: values.to_numpy()}, index=embed_plot.obs_names),
        obsm={"X_umap": embed_plot.obsm["X_umap"]},
    )

    # No tight_layout here: it collapses the colour bar sc.pl.umap attaches to `ax`.
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    sc.pl.umap(embed_plot, color=group_key, groups=[group], title=f"{group} ({group_key})",
               ax=axes[0], show=False, legend_loc="none", na_in_legend=False,
               size=size, frameon=False)
    sc.pl.umap(panel, color=label, title=label, ax=axes[1], show=False,
               size=size, frameon=False, cmap=drvi.utils.pl.cmap.saturated_sky_cmap,
               vmin=vmin, vcenter=0, vmax=vmax)

    # The number behind the picture: the same contrast the heatmap shows as one cell.
    in_group = embed_plot.obs[group_key].astype(str).to_numpy() == str(group)
    print(f"{label}: {values[in_group].mean():+.2f} mean over the {int(in_group.sum()):,} "
          f"cells of {group_key}={group} vs {values[~in_group].mean():+.2f} over the other "
          f"{int((~in_group).sum()):,}")
    return fig

### Which factor encodes which group: SMI

Picking the pairs by how *far* a group sits along an axis answers the wrong question: a program
shared by several related groups can put all of them far out, and the panel then shows the
compartment rather than the group. That failure mode is the normal case here, where every
group is a state of one tumour compartment. The DRVI tutorial scores the thing we actually
want - a **one-to-one correspondence** between a factor direction and a group - with the Scaled
Mutual Information, normalised to `[0, 1]`.

Three steps, taken from the tutorial:

- `build_directional_df` turns the latent space into a *cells x factor-directions* matrix.
  Each dimension is split in two, the vanished halves are dropped, and each half is **clipped
  at zero** (`pos = z.clip(min=0)`, `neg = -z.clip(max=0)`): the positive half reads zero
  wherever the factor is negative and vice versa, so the two halves become two independent
  variables, each switched off outside its own semiaxis.
- `smi_matches` hands that matrix to `DiscreteDisentanglementBenchmark` with `metrics=["SMI"]`.
  Internally each direction is binned into 10 uniform bins and the mutual information between
  the binned factor and the one-hot indicator of each group is divided by that group's entropy
  - the normalisation that makes a rare group and an abundant one comparable.
- `store_matches` writes the best match per direction back into `embed.var` as
  `positive_direction_match_with_<suffix>` / `negative_direction_match_with_<suffix>` and the
  whole table into `embed.uns`, the layout the other notebooks of the tutorial series read.

**Run twice, against two different targets.** `cell_type` is constant here, so the tutorial's
own target does not exist on this object and SMI against it is zero by construction. The two
substitutes answer two different questions:

| target | question | how to read a hit |
|---|---|---|
| `optscib_tum_leiden` | which factor reproduces a cluster of *these* cells? | a tumour state, on the subset's own terms. Its clusters come from the expression graph; only their *number* was picked against the label below |
| `cell_type_01_4` | which factor reproduces the normal-breast label these cells were *mistaken for*? | a landmark: the tumour state has a normal counterpart the model can name. Never evidence that the cell is that normal type |

A direction that scores on both is the strongest result this section can produce; one that
scores only on leiden is a state the normal-breast vocabulary has no word for, which is what
phase 05 exists to find.

The annotation never enters DRVI's training, so it cannot bias the factors: it only names them
afterwards. What SMI measures is the agreement between a factor and *a grouping* - not between
a factor and the truth - so a mediocre score can be the grouping's fault as much as the
factor's, and the assignments are worth checking against each dimension's own panel below.

In [ ]:
# Imports local to this section, so the Libraries cell at the top does not have to be re-run:
# `networkx` and the benchmark class are only used here.
import networkx as nx
from drvi.utils.metrics import DiscreteDisentanglementBenchmark


def build_directional_df(embed):
    """Cells x (non-vanished factor-directions). Column names carry a '+'/'-' suffix."""
    embed_pos = embed[:, ~embed.var["vanished_positive_direction"]].copy()
    embed_neg = embed[:, ~embed.var["vanished_negative_direction"]].copy()
    embed_pos.var.index = embed_pos.var["title"] + "+"
    embed_neg.var.index = embed_neg.var["title"] + "-"
    embed_pos.X = embed_pos.X.clip(min=0)
    embed_neg.X = -embed_neg.X.clip(max=0)
    return pd.concat([embed_pos.to_df(), embed_neg.to_df()], axis=1).loc[embed.obs.index]


def smi_matches(embed_directional_df, target, threshold):
    """SMI between every factor-direction and every category of `target`.

    Returns (full SMI matrix, long table of pairs with SMI >= threshold sorted descending).
    """
    benchmark = DiscreteDisentanglementBenchmark(
        embed_directional_df.values,
        dim_titles=embed_directional_df.columns,
        discrete_target=target,
        metrics=["SMI"],
        aggregation_methods=[],
    )
    benchmark.evaluate()
    smi = benchmark.get_results_details()["SMI"]
    smi.index.name = "title"

    top = (
        smi.reset_index()
        .melt(id_vars="title", value_vars=smi.columns)
        .query("value >= @threshold")
        .reset_index(drop=True)
        .sort_values("value", ascending=False)
    )
    return smi, top


def store_matches(embed, top, suffix):
    """Store the best match per factor-direction in `embed.var` and the full table in `.uns`."""
    first = top.drop_duplicates(subset=["title"]).copy()
    first["direction"] = first["title"].str[-1:]
    first["title"] = first["title"].str[:-1]

    embed.var.set_index("title", drop=False, inplace=True)
    for d, sign in [("positive", "+"), ("negative", "-")]:
        col = f"{d}_direction_match_with_{suffix}"
        embed.var[col] = None
        sub = first.query("direction == @sign")
        embed.var.loc[sub["title"], col] = sub["variable"].values
    embed.var.index = embed.var["original_dim_id"].astype(int).astype(str)
    embed.var.index.name = None

    embed.uns[f"best_smi_matching_{suffix}_results"] = top


def plot_packed_network(df, title_col="title", var_col="variable", val_col="value",
                        figsize=(20, 20)):
    """Visualize factor-group associations as a network with edge weights."""
    G = nx.from_pandas_edgelist(df, title_col, var_col, edge_attr=val_col)

    pos = {}
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    cols = 3
    for i, nodes in enumerate(components):
        sub_pos = nx.spring_layout(G.subgraph(nodes), weight=val_col, k=0.5)
        r, c = divmod(i, cols)
        for n, (x, y) in sub_pos.items():
            pos[n] = (x + c * 3, y - r * 3)

    plt.figure(figsize=figsize)
    titles = set(df[title_col])
    weights = [d[val_col] for u, v, d in G.edges(data=True)]
    nx.draw(
        G, pos,
        with_labels=True, font_size=8, font_weight="bold", node_size=600,
        node_color=["#A0CBE2" if n in titles else "#FF9E9E" for n in G.nodes()],
        width=[w * 4 for w in weights],
        edge_color=weights, edge_cmap=plt.cm.Oranges, alpha=0.6,
    )
    edge_labels = {(u, v): f"{d[val_col]:.2f}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)
    plt.axis("off")

In [ ]:
# The tutorial's default threshold is 0.7, and in 02_2 it selects ten clean one-to-one matches
# on the whole dataset. In 04_2, on a single lineage, it selected nothing (the best pair
# reached 0.65) and was lowered to look at partial correspondences instead. This subset is one
# compartment further down and its groups are states rather than cell types, so the same 0.4
# is the starting point; raise it if the table below is long and flat.
SMI_THRESHOLD = 0.4

# ~36k cells: seconds, against the ~11 minutes the same cell takes on the 620k cells of 02_2.
embed_directional_df = build_directional_df(embed)
print(f"{embed_directional_df.shape[1]} non-vanished factor directions "
      f"out of {2 * embed.n_vars}")
embed_directional_df.iloc[:3, :5]

In [ ]:
# The two targets, run through the same screening. `cell_type` is deliberately not among
# them: it is constant on this object, and SMI against a constant is zero everywhere.
SMI_TARGETS = {"leiden": LEIDEN_KEY} if HAS_LEIDEN else {}
SMI_TARGETS["celltypist_01_4"] = GROUP_KEY

smi_similarity, smi_top_matches = {}, {}
for suffix, key in SMI_TARGETS.items():
    smi_similarity[suffix], smi_top_matches[suffix] = smi_matches(
        embed_directional_df, embed.obs[key], SMI_THRESHOLD
    )
    print(f"{suffix:>16} ({key}): SMI matrix {smi_similarity[suffix].shape}, "
          f"{len(smi_top_matches[suffix])} pairs >= {SMI_THRESHOLD}, "
          f"best {smi_similarity[suffix].to_numpy().max():.3f}")

smi_top_matches[list(SMI_TARGETS)[0]].head(20)

In [ ]:
# The matches as a bipartite graph, one per target: the components that are not a single edge
# are the interesting ones - one group carried by two factors, or one factor shared by two
# groups.
for suffix, top in smi_top_matches.items():
    if top.empty:
        print(f"[skip] {suffix}: no pair above {SMI_THRESHOLD}")
        continue
    plot_packed_network(top, figsize=(20, 20))
    savefig(f"smi_{suffix}_network")
    plt.show()

In [ ]:
# Best match per direction into embed.var, whole table into embed.uns, as the tutorial does so
# that a later notebook can pick the assignments up. The suffix keeps the two targets apart:
# `..._match_with_celltypist_01_4` and `..._match_with_leiden`.
for suffix, top in smi_top_matches.items():
    if top.empty:
        continue
    store_matches(embed, top, suffix=suffix)

[c for c in embed.var.columns if "direction_match_with" in c]

In [ ]:
# A copy of the tables in TABLE_DIR. The tutorial writes `embed` back to disk here;
# LATENT_H5AD is a few hundred MB and only these columns changed, so the rewrite is left
# opt-in.
for suffix, top in smi_top_matches.items():
    path = TABLE_DIR / f"smi_{suffix}_matches_{RUN_ID}.csv"
    top.to_csv(path, index=False)
    print(f"[write] {path} ({len(top)} rows)")

WRITE_BACK_EMBED = False
if WRITE_BACK_EMBED:
    ad.settings.allow_write_nullable_strings = True
    embed.write_h5ad(LATENT_H5AD)
    print(f"[write] {LATENT_H5AD}")

### The pairs, one at a time

Every match above, as the two-panel figure: the group on the left, the factor direction SMI
paired it with on the right. A high SMI says the two carry the same information; the figure
says whether they carry it in the same *place*, which is the part a number cannot show - and
the further the scores are from 1, the more it is the figure that decides whether a pair is
worth anything.

The file name carries the target, so a `cell_type_01_4` pair and a leiden pair for the same
dimension sit side by side instead of overwriting each other.

In [ ]:
# `title` carries the direction as its last character: "DR 18-" -> ("DR 18", "-").
for suffix, top in smi_top_matches.items():
    group_key = SMI_TARGETS[suffix]
    for row in top.itertuples(index=False):
        dim_title, direction, group = row.title[:-1], row.title[-1], row.variable
        print(f"[{suffix}] SMI {row.value:.3f}", end="  ")
        fig = plot_dim_vs_group(group, dim_title, direction, group_key=group_key)
        savefig(f"dim_vs_{suffix}_{group}_{dim_title.replace(' ', '')}{direction}", fig=fig)
        plt.show()

## What the factors are doing: biological processes

SMI above answers "which factor *is* which group", and it can only ever answer it for the
groups this subset has: a clustering of these cells, and a normal-breast label that is a
landmark rather than an identity. Every direction it leaves unnamed is still a candidate for
something - a factor without a group match is not necessarily empty, it can be a **process**
running across several groups (secretion, interferon response, cell cycle, stress), which is
exactly the thing SMI is built not to reward.

**This section carries more weight here than it does in 04_2.** This phase has no non-constant
biological annotation of its own, so naming a dimension **by its genes** is the only reading
that owes nothing to a borrowed vocabulary. The SMI matches above are the cross-check on it,
not the evidence. It is the same argument `05_7_factor_first` makes for Route B, run here at
the level of "what is this axis about" rather than "does it enrich for the collaborator's
lists".

This is the DRVI tutorial *Identifying biological processes of DRVI factors*, run on this
notebook's factors. Three non-LLM tools, all reading the same per-gene interpretability scores
computed above:

| Tool | Method | Input | What it adds |
| --- | --- | --- | --- |
| **Enrichr** (via GSEApy) | over-representation (ORA) | top-N gene list | fast, the largest library collection |
| **g:Profiler** | over-representation (ORA) | ordered gene query | g:SCS correction, built for the GO hierarchy |
| **decoupler** | activity inference (ULM/MLM) | gene-score matrix + prior network | reports *regulators* (TFs), not terms |

Three things to keep in mind, none of them a departure from the tutorial:

- **The background is the HVG panel, 2,000 genes.** ORA asks "is this term over-represented in
  the top genes *relative to the universe those genes were drawn from*", and that universe is
  whatever DRVI was trained on: the interpretability table has one row per model gene and no
  others, so there is no wider list to test against. The tutorial does the same
  (`gene_background = adata.var_names.tolist()`); what changes here is how that panel was
  chosen - 2,000 genes selected in `05_2` **inside the tumour**, so the terms found are found
  among the genes that vary most between malignant cells, not between tumour and normal. It is
  a ceiling on coverage, not a bias: a process whose genes are mostly outside the panel cannot
  be found. Letting the tools fall back to their whole-genome default would be the wrong call -
  it would count the HVG selection itself as enrichment.
- **The scores are the OOD ones** (`OOD_combined`), which favour genes *specific* to a
  direction. `IND_linear_weighted_mean` is the alternative and needs a higher cutoff, ~0.5
  against ~0.1 - see the config cell.
- **Pre-ranked GSEA is not run.** DRVI's scores are non-negative by construction, one ranking
  per factor *direction*, so there is no signed bottom of the list for GSEA to use. The
  tutorial makes the same call: ORA plus TF activity, no GSEA.

Everything here is guiding. A term is a hypothesis about a factor, to be read next to the
factor's top genes and the groups it lights up, not a conclusion.

### Setup

Three packages on top of the environment of this phase: `gseapy` (1.3.0), `gprofiler-official`
(1.0.0) and `decoupler` (2.2.0), all three already in `benchmark-py-r`. On any other
environment they are:

```
pip install gseapy gprofiler-official "decoupler>=2.0"
```

Both ORA tools query a web service (Enrichr for the library, g:Profiler for the whole run), so
these cells need network and are the slow ones of the notebook. The imports are local to the
section, as in the SMI one, so the Libraries cell at the top does not have to be re-run.

In [35]:
# Local imports: only this section uses them.
import gseapy
from gprofiler import GProfiler
import decoupler as dc
from statsmodels.stats.multitest import multipletests

In [36]:
SCORE_KEY = "OOD_combined"   # or "IND_linear_weighted_mean", with the higher cutoffs below

# The gene x factor-direction table every tool below reads.
scores_df = model.get_interpretability_scores(embed, adata, key=SCORE_KEY)

# The universe the top genes are drawn from: the 2,000 HVGs of INPUT_H5AD, nothing else.
gene_background = adata.var_names.tolist()

print(f"{scores_df.shape[0]:,} genes x {scores_df.shape[1]} factor directions, key={SCORE_KEY!r}")
print(f"background  {len(gene_background):,} genes (the HVG panel of 05_2, selected inside the tumour)")

2,000 genes x 62 factor directions, key='OOD_combined'
background  2,000 genes (the HVG panel of 05_2, selected inside the tumour)


In [37]:
def top_genes(scores_df, col, cutoff, top_n):
    """Genes of a factor direction with score >= cutoff, at most `top_n`, best first."""
    s = scores_df[col]
    return s[s >= cutoff].nlargest(top_n).index.astype(str).tolist()


def factor_first(df):
    """`df` with the `factor` column moved to the front (no-op if empty or absent)."""
    if df.empty or "factor" not in df.columns:
        return df
    return df[["factor", *df.columns.drop("factor")]]


# How many directions carry a gene list at all, at the cutoff used below: a direction whose
# best gene sits under the cutoff has no program to test and is skipped by every tool.
GENE_CUTOFF = 0.1 if SCORE_KEY == "OOD_combined" else 0.5   # ~0.1 for OOD, ~0.5 for IND
TOP_N = 100                                              # max genes per direction

sizes = pd.Series(
    {c: len(top_genes(scores_df, c, GENE_CUTOFF, TOP_N)) for c in scores_df.columns},
    name="n_genes",
)
print(f"directions with at least one gene past {GENE_CUTOFF}: "
      f"{(sizes > 0).sum()} / {len(sizes)}")
print(f"median list length among those: {int(sizes[sizes > 0].median()) if (sizes > 0).any() else 0} genes")
sizes.sort_values(ascending=False).head(10)

directions with at least one gene past 0.1: 53 / 62
median list length among those: 86 genes


DR 1-     100
DR 2+     100
DR 3-     100
DR 5-     100
DR 4-     100
DR 4+     100
DR 7-     100
DR 6+     100
DR 10+    100
DR 11+    100
Name: n_genes, dtype: int64

### 1. Enrichr (via GSEApy)

The plain over-representation test: take the top genes of a direction, ask whether any term of
a library is over-represented among them against the background, Benjamini-Hochberg over the
terms of that direction.

`GO_Biological_Process_2023` is the tutorial's library. The appendix of the tutorial lists the
swap-ins - `MSigDB_Hallmark_2020` for broad states, `Reactome_Pathways_2024`, `KEGG_2026`,
`CellMarker_2024` and `PanglaoDB_Augmented_2021` for markers - and only `GSEAPY_DB` has to
change to use one. The library is fetched from Enrichr the first time, so this cell needs
network.

In [38]:
GSEAPY_DB = "GO_Biological_Process_2023"   # any Enrichr library, see the tutorial appendix
PADJ_THRESHOLD = 0.05


def run_gseapy_enrichr(scores_df, gene_sets, cutoff, top_n, padj_cutoff, background):
    rows = []
    for col in scores_df.columns:
        genes = top_genes(scores_df, col, cutoff, top_n)
        if not genes:
            continue
        try:
            enr = gseapy.enrich(gene_list=genes, gene_sets=gene_sets, background=background,
                                no_plot=True, outdir=None)
        except Exception as e:
            print(f"ORA failed for {col}: {e}")
            continue
        hits = enr.results[enr.results["Adjusted P-value"] < padj_cutoff]
        rows.append(hits.assign(factor=col))
    return factor_first(pd.concat(rows, ignore_index=True)) if rows else pd.DataFrame()


enrichr_results = run_gseapy_enrichr(
    scores_df, GSEAPY_DB, GENE_CUTOFF, TOP_N, PADJ_THRESHOLD, gene_background
)
n_sig = enrichr_results["factor"].nunique() if not enrichr_results.empty else 0
print(f"Enrichr: {n_sig} / {scores_df.shape[1]} factor directions with at least one term "
      f"at padj < {PADJ_THRESHOLD}")
enrichr_results.head()

Enrichr: 39 / 62 factor directions with at least one term at padj < 0.05


,factor,Gene_set,Term,P-value,Adjusted P-value,Old P-value,Old adjusted P-value,Odds Ratio,Combined Score,Genes
0,DR 1-,GO_Biological_Process_2023,Extracellular Matrix Organization (GO:0030198),0.0,0.0,0,0,11.873016,270.021186,POSTN;COL15A1;LUM;MMP1;MMP2;COL11A1;DPT;COL1A1...
1,DR 1-,GO_Biological_Process_2023,External Encapsulating Structure Organization ...,0.0,0.0,0,0,14.793103,308.359706,POSTN;COL15A1;MMP1;MMP2;COL11A1;COL1A1;MMP11;C...
2,DR 1-,GO_Biological_Process_2023,Extracellular Structure Organization (GO:0043062),0.0,0.0,0,0,14.793103,308.359706,POSTN;COL15A1;MMP1;MMP2;COL11A1;COL1A1;MMP11;C...
3,DR 1-,GO_Biological_Process_2023,Collagen Fibril Organization (GO:0030199),0.0,0.000001,0,0,54.985507,1058.971102,COL1A1;COL3A1;COL1A2;COL5A1;LUM;COL11A1;COL5A2...
4,DR 1-,GO_Biological_Process_2023,Regulation Of Cell Migration (GO:0030334),0.000094,0.016434,0,0,3.906977,36.234699,PDGFRB;TNXB;THY1;IGF1;LDB2;SULF1;COL1A1;CLDN5;...


In [39]:
# One row per factor direction: its single most significant term.
enrichr_top = (
    enrichr_results.sort_values("Adjusted P-value")
    .drop_duplicates(subset=["factor"], keep="first")
    [["factor", "Term", "Adjusted P-value", "Odds Ratio", "Genes"]]
    if not enrichr_results.empty else pd.DataFrame()
)
enrichr_top

,factor,Term,Adjusted P-value,Odds Ratio,Genes
1127,DR 29+,Defense Response To Symbiont (GO:0140546),0.0,38.206993,IFITM3;IFITM1;IFITM2;IFI6;DDX60L;IFIT1;DDX60;S...
298,DR 7-,Mitotic Sister Chromatid Segregation (GO:0000070),0.0,76.274725,PLK1;KIF14;CDCA8;NCAPG;KIF23;KIF22;KNSTRN;SMC4...
1211,DR 31-,Cellular Response To Zinc Ion (GO:0071294),0.0,inf,MT2A;MT1A;MT1M;MT1F;MT1G;MT1X;MT1H;MT1E
174,DR 5-,DNA Metabolic Process (GO:0006259),0.0,37.996337,FEN1;DUT;BLM;RNASEH2A;RFC4;MCM7;RFC2;HMGB2;NUD...
981,DR 28+,Mitotic Sister Chromatid Segregation (GO:0000070),0.0,52.527778,CDCA5;PLK1;KIF14;CDCA8;NCAPG;KIF23;KIF11;SMC4;...
423,DR 9-,Regulation Of Transcription By RNA Polymerase ...,0.0,8.070588,CEBPD;CITED2;GATA3;TNF;ZFP36;MYC;HES1;JUNB;IER...
837,DR 23+,Antigen Processing And Presentation Of Exogeno...,0.0,100.9375,HLA-DMA;HLA-DRB5;HLA-DRA;HLA-A;HLA-F;HLA-DQA2;...
1204,DR 30-,Intermediate Filament Organization (GO:0045109),0.0,78.153558,KRT81;KRT17;KRT16;KRT15;KRT14;KRT23;PKP1;KRT5;...
642,DR 16+,Defense Response To Virus (GO:0051607),0.0,11.365854,GBP5;IFITM1;CD40;STAT1;NLRC5;SAMHD1;TNF;IFIT3;...
0,DR 1-,Extracellular Matrix Organization (GO:0030198),0.0,11.873016,POSTN;COL15A1;LUM;MMP1;MMP2;COL11A1;DPT;COL1A1...


**How to read this.** ORA is only as good as the library and as the gene list. A direction whose
biology is well covered by GO returns terms that are specific and internally consistent (a
secretory factor returning secretion / hormone terms, not one secretion term surrounded by
noise); a direction whose biology is not covered returns whatever loosely overlaps. The check is
the g:Profiler run below: a term that both ORA tools return, with different multiple-testing
machinery, is worth more than a term either returns alone.

One expectation specific to this subset: **the cell cycle will be there.** `Lumsec-prol` is 51%
of these cells, `phase` is not constant, and proliferation is the most reliably enriched process
in any tumour compartment. A dimension whose top terms are mitotic is a real finding about the
axis - and it is also the confound `05_6` recomputes its target region inside G1 to measure. Read
those two together.

### 2. g:Profiler

The same over-representation question, two things done differently. The correction is **g:SCS**
rather than BH: GO terms are nested, so the tests are not independent and BH is anti-conservative
on them; g:SCS is calibrated on random queries against the actual GO graph. And the query is
**ordered** - the ranked gene list is walked and the best-enriched prefix is kept - which suits a
continuous score better than a hard top-100 cut.

Coverage is usually lower than Enrichr's. That is the point of running both.

In [40]:
ORGANISM = "hsapiens"
GP_SOURCE = ["GO:BP"]        # e.g. ["GO:MF"], ["REAC"], ["KEGG"], ["HP"]
PVAL_THRESHOLD = 0.05


def run_gprofiler(scores_df, background, organism, sources, pval_threshold, cutoff, top_n):
    gp = GProfiler(return_dataframe=True)
    rows = []
    for col in scores_df.columns:
        genes = top_genes(scores_df, col, cutoff, top_n)
        if not genes:
            continue
        res = gp.profile(organism=organism, query=genes, sources=sources, ordered=True,
                         user_threshold=pval_threshold, background=background)
        if res is None or res.empty:
            continue
        rows.append(res.assign(factor=col))
    return factor_first(pd.concat(rows, ignore_index=True)) if rows else pd.DataFrame()


gprofiler_results = run_gprofiler(
    scores_df, gene_background, ORGANISM, GP_SOURCE, PVAL_THRESHOLD, GENE_CUTOFF, TOP_N
)
if not gprofiler_results.empty:
    # `parents` is a list per row and h5ad cannot store that, so it is flattened now.
    gprofiler_results["parents"] = gprofiler_results["parents"].astype(str)

n_sig = gprofiler_results["factor"].nunique() if not gprofiler_results.empty else 0
print(f"g:Profiler: {n_sig} / {scores_df.shape[1]} factor directions with at least one term "
      f"at p < {PVAL_THRESHOLD}")
gprofiler_results.sort_values(["factor", "p_value"]).head() if not gprofiler_results.empty else None

g:Profiler: 14 / 62 factor directions with at least one term at p < 0.05


,factor,source,native,name,p_value,significant,description,term_size,query_size,intersection_size,effective_domain_size,precision,recall,query,parents
0,DR 1-,GO:BP,GO:0030199,collagen fibril organization,2.392784e-03,True,"""Any process that determines the size and arra...",23,12,7,1865,0.583333,0.304348,query_1,['GO:0030198']
84,DR 14+,GO:BP,GO:0090131,mesenchyme migration,3.872228e-02,True,"""The process in which the population of cells ...",3,9,3,1865,0.333333,1.000000,query_1,"['GO:0072132', 'GO:0090130']"
85,DR 19+,GO:BP,GO:0042026,protein refolding,2.082722e-05,True,"""The process carried out by a cell that restor...",7,9,6,1865,0.666667,0.857143,query_1,['GO:0006457']
86,DR 20-,GO:BP,GO:0006953,acute-phase response,1.493227e-02,True,"""An acute inflammatory response that involves ...",19,42,9,1865,0.214286,0.473684,query_1,['GO:0002526']
87,DR 23+,GO:BP,GO:0002478,antigen processing and presentation of exogeno...,1.634841e-10,True,"""The process in which an antigen-presenting ce...",17,14,10,1865,0.714286,0.588235,query_1,"['GO:0019884', 'GO:0048002']"


In [41]:
# One row per factor direction: its single most significant term.
gprofiler_top = (
    gprofiler_results.sort_values("p_value")
    .drop_duplicates(subset=["factor"], keep="first")
    [["factor", "native", "name", "p_value", "term_size", "query_size", "intersection_size"]]
    if not gprofiler_results.empty else pd.DataFrame()
)
gprofiler_top

,factor,native,name,p_value,term_size,query_size,intersection_size
149,DR 31-,GO:1990169,stress response to copper ion,4.198983e-14,8,8,8
100,DR 25+,GO:0002250,adaptive immune response,5.473867e-14,185,71,64
125,DR 29+,GO:0045071,negative regulation of viral genome replication,2.508731e-11,24,34,15
8,DR 7-,GO:0000070,mitotic sister chromatid segregation,4.754577e-11,50,87,37
59,DR 8+,GO:0006119,oxidative phosphorylation,1.247373e-10,27,13,13
87,DR 23+,GO:0002478,antigen processing and presentation of exogeno...,1.634841e-10,17,14,10
103,DR 28+,GO:0007059,chromosome segregation,6.758300e-08,84,92,52
140,DR 30-,GO:0045109,intermediate filament organization,2.055025e-06,16,9,8
85,DR 19+,GO:0042026,protein refolding,2.082722e-05,7,9,6
1,DR 5-,GO:0006260,DNA replication,5.736120e-04,38,43,23


**How to read this.** The hits of one factor tend to organise general-to-specific, and the
specific end is where g:Profiler earns its place: it can name the mechanism where Enrichr names
the compartment. Read the two tables side by side - the summary further down does exactly that -
and trust the convergent calls.

### 3. decoupler

A different question from the two above. ORA asks whether a gene list overlaps a term; decoupler
takes the whole score vector of a direction and asks which **regulator** would explain it, by
scoring each TF regulon of a prior network with a linear model (ULM) and a multivariate one
(MLM), then taking the consensus. A significant hit is a transcription factor whose targets carry
the factor's program - a mechanistic layer neither ORA tool can give.

The networks come from OmniPath: **CollecTRI** (comprehensive TF to target regulons, the default
here), **DoRothEA** (the same with confidence tiers A-D), **PROGENy** (pathway footprints, for
signalling rather than TFs).

The 2,000-gene panel bites hardest here. A regulon is only usable if enough of its targets
survived HVG selection, so the network is intersected with the panel before anything is inferred
and `DC_MIN` decides how many targets are enough. The cell prints what survives; if that number
is small, lower `DC_MIN` to 5 before reading anything into the coverage.

In [42]:
DC_GENESET = "collectri"     # or "dorothea", "progeny"
DC_ORGANISM = "human"
DC_CUTOFF = 0.01 if SCORE_KEY == "OOD_combined" else 0.05   # scores under this read as zero
FDR_THRESHOLD = 0.05
DC_METHODS = ["ulm", "mlm"]
DC_MIN = 10                  # min targets of a regulon present in the HVG panel
DOROTHEA_LEVELS = ["A", "B", "C"]
FDR_METHOD = "fdr_bh"

net_dispatch = {
    "collectri": lambda: dc.op.collectri(organism=DC_ORGANISM),
    "dorothea": lambda: dc.op.dorothea(organism=DC_ORGANISM, levels=DOROTHEA_LEVELS),
    "progeny": lambda: dc.op.progeny(organism=DC_ORGANISM),
}
net = net_dispatch.get(
    DC_GENESET.strip().lower(),
    lambda: dc.op.resource(name=DC_GENESET, organism=DC_ORGANISM),
)()
cols = ["source", "target"] + (["weight"] if "weight" in net.columns else [])
net = net[cols].dropna().drop_duplicates().reset_index(drop=True)
print(f"network     {len(net):,} interactions, {net['source'].nunique():,} regulators")

# What is left of it once the HVG panel is the only vocabulary available.
panel = set(g.upper() for g in gene_background)
in_panel = net[net["target"].astype(str).str.upper().isin(panel)]
usable = in_panel.groupby("source").size()
print(f"on the panel {len(in_panel):,} interactions, {in_panel['source'].nunique():,} regulators")
print(f"usable      {int((usable >= DC_MIN).sum()):,} regulators with >= {DC_MIN} targets in the panel")

network     42,990 interactions, 1,185 regulators
on the panel 12,016 interactions, 889 regulators
usable      289 regulators with >= 10 targets in the panel


In [43]:
def run_decouple(factors_by_genes, net, methods, tmin, fdr_method):
    mat = factors_by_genes.copy()
    mat.columns = mat.columns.astype(str).str.upper()

    net_u = net.copy()
    net_u["target"] = net_u["target"].astype(str).str.upper()

    keep = mat.columns.intersection(net_u["target"].unique())
    mat = mat[keep].replace([np.inf, -np.inf], 0.0).fillna(0.0)

    res = dc.mt.decouple(data=mat, net=net_u, methods=methods, cons=False, tmin=tmin)
    _, pvals = dc.mt.consensus(res)

    out = pvals.stack().reset_index(name="p_value").rename(
        columns={"level_0": "factor", "level_1": "term"}
    )
    out["p_adj"] = multipletests(out["p_value"].values, method=fdr_method)[1]
    return out


# Rows are factor directions, columns genes: the transpose of the score table, with the noise
# floor zeroed so a regulon is not scored on numerically meaningless values.
input_df = scores_df.copy()
input_df[input_df < DC_CUTOFF] = 0
decoupler_all = run_decouple(input_df.T, net, DC_METHODS, DC_MIN, FDR_METHOD)

# The most significant regulator per direction, for the summary view.
decoupler_top = (
    decoupler_all[decoupler_all["p_adj"] < FDR_THRESHOLD]
    .sort_values("p_adj")
    .groupby("factor", as_index=False)
    .first()
)
print(f"decoupler: {decoupler_top['factor'].nunique()} / {scores_df.shape[1]} factor directions "
      f"with a regulator at FDR < {FDR_THRESHOLD}")
decoupler_top.sort_values("p_adj")

decoupler: 22 / 62 factor directions with a regulator at FDR < 0.05


,factor,term,p_value,p_adj
21,DR 9+,HOXA10,3.193741e-19,5.537946e-15
10,DR 19+,HSF2,4.119384e-17,3.571506e-13
13,DR 22+,KDM5B,6.436950e-17,3.720557e-13
6,DR 15-,RUNX2,4.133761e-13,1.433588e-09
11,DR 20-,NR5A2,1.686750e-10,4.874709e-07
18,DR 30+,POU2F2,6.716314e-09,1.455761e-05
2,DR 12+,RUNX2,3.476473e-08,6.698005e-05
17,DR 26-,NR2F2,6.153729e-08,1.067057e-04
4,DR 14+,SRF,2.664332e-07,4.199956e-04
15,DR 23-,NFIL3,3.743667e-07,5.409598e-04


**How to read this.** Where decoupler and the ORA tools agree the reading is easy - a TF whose
biology is the term already returned. Where decoupler alone speaks, the hit is a candidate driver
to check against the direction's top genes, not a conclusion: with a 2,000-gene vocabulary a
regulon is represented by a handful of targets, and a TF can score high because its few visible
targets happen to be the factor's markers. Coverage is limited by design too - most factors are
not the work of one dominant TF.

### The three tools next to SMI

The tutorial ends by pulling every tool's output into one per-factor view, in a separate curation
notebook. That view is small enough to build here: one row per factor direction, its SMI match
against **each** of this phase's two targets, the top hit of each of the three tools, and its top
genes. It is the table to read this section from, and the one written to disk below.

Two SMI columns rather than 04_2's one, for the reason the SMI section gives: `smi_leiden` is a
match against a partition of these cells, `smi_celltypist_01_4` against a normal-breast landmark.
A direction with a leiden match **and** convergent terms is the strongest row this notebook can
produce. A direction with neither and convergent terms is a process factor - which is what this
section was run to find.

**Reading the empty cells.** A large part of the table is blank, and that is structure, not
failure. A *dimension* is split into two directions here, but a DRVI factor is one-sided: it
encodes a program in one direction and sits at baseline in the other. The silent halves have no
gene above the cutoff, so no ORA runs on them and their row is blank by construction;
`has_program` marks the difference and the cell below counts them for this run rather than
quoting 04's numbers. Two consequences:

- **decoupler ignores that distinction.** It reads the whole score vector, noise floor included,
  so it can return a regulator for a direction that has no program at all - a significant p-value
  on numerically empty input. Those hits are noise; the summary prints how many there are.
- **SMI ignores it too.** A silent half can still correlate with a group. An SMI match on a
  direction with `has_program == False` is the pattern to distrust first.

In [44]:
def _first(df, key_col, out_col, rename):
    if df is None or df.empty:
        return pd.Series(dtype=object, name=rename)
    return df.set_index(key_col)[out_col].rename(rename)


factor_summary = pd.DataFrame(index=pd.Index(scores_df.columns, name="factor"))

# One pair of columns per SMI target, kept apart on purpose: a leiden match and a CellTypist
# match are not the same kind of statement about a factor.
for suffix, top in smi_top_matches.items():
    best = top.drop_duplicates(subset=["title"]).set_index("title") if not top.empty else None
    factor_summary[f"smi_{suffix}"] = best["variable"] if best is not None else pd.NA
    factor_summary[f"smi_{suffix}_value"] = best["value"].round(3) if best is not None else pd.NA

smi_label_cols = [f"smi_{s}" for s in smi_top_matches]
factor_summary["any_smi"] = factor_summary[smi_label_cols].notna().any(axis=1)

factor_summary["enrichr_term"] = _first(enrichr_top, "factor", "Term", "enrichr_term")
factor_summary["enrichr_padj"] = _first(enrichr_top, "factor", "Adjusted P-value", "enrichr_padj")
factor_summary["gprofiler_term"] = _first(gprofiler_top, "factor", "name", "gprofiler_term")
factor_summary["gprofiler_p"] = _first(gprofiler_top, "factor", "p_value", "gprofiler_p")
factor_summary["decoupler_tf"] = _first(decoupler_top, "factor", "term", "decoupler_tf")
factor_summary["decoupler_padj"] = _first(decoupler_top, "factor", "p_adj", "decoupler_padj")
factor_summary["n_tools"] = factor_summary[
    ["enrichr_term", "gprofiler_term", "decoupler_tf"]].notna().sum(axis=1)
factor_summary["top_genes"] = [
    ", ".join(top_genes(scores_df, c, GENE_CUTOFF, 8)) for c in scores_df.columns
]

# Does the direction have a program at all? The silent halves have no gene above the cutoff, so
# no ORA runs on them - but decoupler does, because it reads the whole score vector, and it can
# call a regulator on what is numerically noise. The flag keeps those hits visible instead of
# letting them look like findings.
factor_summary["max_score"] = scores_df.max()
factor_summary["has_program"] = factor_summary["max_score"] >= GENE_CUTOFF

# Unlabelled first, then by how many tools agree: the process candidates float to the top.
factor_summary = factor_summary.sort_values(
    ["any_smi", "n_tools"], ascending=[True, False])

n_dirs = len(factor_summary)
print(f"{int(factor_summary['any_smi'].sum())} of {n_dirs} directions carry an SMI match "
      f"(any target), {int(factor_summary['has_program'].sum())} carry a gene program, "
      f"{int((factor_summary['n_tools'] > 0).sum())} have at least one enrichment hit")
print(f"{int(((~factor_summary['any_smi']) & (factor_summary['n_tools'] >= 2)).sum())} "
      f"unlabelled directions with two tools or more: the process candidates")
no_prog = factor_summary[~factor_summary["has_program"]]
print(f"on the {len(no_prog)} directions with NO program: "
      f"{int(no_prog['decoupler_tf'].notna().sum())} decoupler regulators (read as noise) and "
      f"{int(no_prog['any_smi'].sum())} SMI matches (distrust these first)")
factor_summary.head(20)

4 of 62 directions carry an SMI match (any target), 53 carry a gene program, 49 have at least one enrichment hit
19 unlabelled directions with two tools or more: the process candidates
on the 9 directions with NO program: 2 decoupler regulators (read as noise) and 0 SMI matches (distrust these first)


,smi_leiden,smi_leiden_value,smi_celltypist_01_4,smi_celltypist_01_4_value,any_smi,enrichr_term,enrichr_padj,gprofiler_term,gprofiler_p,decoupler_tf,decoupler_padj,n_tools,top_genes,max_score,has_program
factor,,,,,,,,,,,,,,,
DR 5-,NaN,NaN,NaN,NaN,False,DNA Metabolic Process (GO:0006259),0.0,DNA replication,5.736120e-04,E2F1,2.656327e-02,3,"HIST1H4C, TYMS, GINS2, CDC45, HIST1H1B, CLSPN,...",7.195954,True
DR 14+,NaN,NaN,NaN,NaN,False,Actin Filament Organization (GO:0007015),0.000272,mesenchyme migration,3.872228e-02,SRF,4.199956e-04,3,"ACTG2, TAGLN, MYL9, ACTA1, CNN1, MYLK, TPM1, C...",10.761212,True
DR 19+,NaN,NaN,NaN,NaN,False,Response To Unfolded Protein (GO:0006986),0.000021,protein refolding,2.082722e-05,HSF2,3.571506e-13,3,"HSPA6, HSPA1A, HSPA1B, HSPA5, PARD6G-AS1, RGS2...",16.386946,True
DR 20-,NaN,NaN,NaN,NaN,False,Negative Regulation Of Endopeptidase Activity ...,0.002905,acute-phase response,1.493227e-02,NR5A2,4.874709e-07,3,"SAA2, SAA1, LBP, CHI3L1, RARRES1, CCL20, SLPI,...",13.292542,True
DR 23+,NaN,NaN,NaN,NaN,False,Antigen Processing And Presentation Of Exogeno...,0.0,antigen processing and presentation of exogeno...,1.634841e-10,RFXAP,3.099012e-03,3,"HLA-DRB5, HLA-DRB1, HLA-DQB1, HLA-DQA1, HLA-DR...",6.566010,True
DR 25+,NaN,NaN,NaN,NaN,False,B Cell Receptor Signaling Pathway (GO:0050853),0.0,adaptive immune response,5.473867e-14,NOTCH1,1.406652e-02,3,"IGLV3-21, IGKV3-20, IGLV1-44, IGHV4-39, IGHV3-...",90.777084,True
DR 4+,NaN,NaN,NaN,NaN,False,Positive Regulation Of Extrinsic Apoptotic Sig...,0.003399,NaN,NaN,TFE3,4.988844e-02,2,"UPP1, PPIF, PMAIP1, IER3, PHLDA1, SFN, HMOX1, ...",2.715262,True
DR 7+,NaN,NaN,NaN,NaN,False,Regulation Of DNA-templated Transcription (GO:...,0.025074,response to corticotropin-releasing hormone,1.616036e-02,NaN,NaN,2,"CTGF, RBKS, AREG, RCAN1, NR4A3, CYR61, NR4A2, ...",1.071896,True
DR 7-,NaN,NaN,NaN,NaN,False,Mitotic Sister Chromatid Segregation (GO:0000070),0.0,mitotic sister chromatid segregation,4.754577e-11,NaN,NaN,2,"CCNB1, CDC20, CCNB2, PLK1, UBE2C, KIF20A, PTTG...",7.669688,True


### Manual exploration

The tutorial closes on one factor read end to end - its `DR 43-`, an interferon response whose
top genes are the canonical ISGs and where all three tools converge on virus terms and on IRF9.
The helper below is the same one, and three factors are put through it here:

- the strongest **unlabelled** candidate the summary just produced - no SMI match, a real gene
  program, the most tools agreeing. This is the process factor the section exists to find;
- the direction with the highest **reconstruction effect**, i.e. the axis the model spent the
  most on, whatever it turned out to be;
- the best **leiden** match, if there is one: a factor that both reproduces a cluster of these
  cells and has a nameable programme is the strongest statement available in this notebook.

Edit the list to walk any other direction.

In [45]:
def show_factor_evidence(factor_label, cutoff=None, n_genes=12):
    """Top genes of a direction and the top hits of the three tools, side by side."""
    cutoff = GENE_CUTOFF if cutoff is None else cutoff
    genes = top_genes(scores_df, factor_label, cutoff, n_genes)
    labels = [f"{s}={factor_summary.loc[factor_label, f'smi_{s}']}"
              for s in smi_top_matches
              if pd.notna(factor_summary.loc[factor_label, f"smi_{s}"])]
    print(f"{factor_label} - SMI: {', '.join(labels) if labels else '(no match above threshold)'}")
    print(f"top genes: {', '.join(genes) if genes else '(none past the cutoff)'}\n")
    for name, df, col in [
        ("Enrichr    ", enrichr_results, "Term"),
        ("g:Profiler ", gprofiler_results, "name"),
        ("decoupler  ", decoupler_all[decoupler_all["p_adj"] < FDR_THRESHOLD], "term"),
    ]:
        terms = (df.loc[df["factor"] == factor_label, col].head(5).tolist()
                 if df is not None and not df.empty else [])
        print(f"{name}: {terms or '(no hit)'}")
    print()


walk = []

# 1. the strongest unlabelled candidate
unlabelled = factor_summary[(~factor_summary["any_smi"])
                            & factor_summary["has_program"]
                            & (factor_summary["n_tools"] > 0)]
if len(unlabelled):
    walk.append(unlabelled.index[0])

# 2. the direction the model spent the most on. `reconstruction_effect` is per DIMENSION, so
#    the direction of it that actually carries a program is the one walked.
top_dim = embed.var.sort_values("reconstruction_effect", ascending=False)["title"].iloc[0]
for d in (f"{top_dim}+", f"{top_dim}-"):
    if d in factor_summary.index and factor_summary.loc[d, "has_program"]:
        walk.append(d)
        break

# 3. the best leiden match, if the run produced one
if "leiden" in smi_top_matches and not smi_top_matches["leiden"].empty:
    walk.append(smi_top_matches["leiden"].iloc[0]["title"])

for d in dict.fromkeys(walk):      # de-duplicated, order kept
    show_factor_evidence(d)

DR 5- - SMI: (no match above threshold)
top genes: HIST1H4C, TYMS, GINS2, CDC45, HIST1H1B, CLSPN, FAM111B, TK1, RRM2, CDC6, PCNA, ASF1B

Enrichr    : ['DNA Metabolic Process (GO:0006259)', 'DNA Repair (GO:0006281)', 'DNA Replication (GO:0006260)', 'Double-Strand Break Repair Via Homologous Recombination (GO:0000724)', 'DNA-templated DNA Replication (GO:0006261)']
g:Profiler : ['DNA replication', 'DNA replication initiation', 'double-strand break repair via break-induced replication', 'nuclear DNA replication', 'DNA-templated DNA replication']
decoupler  : ['E2F1']

DR 1- - SMI: leiden=4
top genes: COL1A2, DCN, SPARC, COL1A1, SFRP2, SPARCL1, AEBP1, LUM, COL3A1, THY1, RARRES2, MMP11

Enrichr    : ['Extracellular Matrix Organization (GO:0030198)', 'External Encapsulating Structure Organization (GO:0045229)', 'Extracellular Structure Organization (GO:0043062)', 'Collagen Fibril Organization (GO:0030199)', 'Regulation Of Cell Migration (GO:0030334)']
g:Profiler : ['collagen fibril organizat

### Save

Same convention as the SMI section: the tables go next to the figures as CSVs, and the write-back
into `LATENT_H5AD` is opt-in. The tutorial stores its results in `embed.uns` and rewrites the
h5ad on the spot; here `embed` is a few hundred MB and nothing else in the file changed, so
`.uns` is filled either way and only the write is left to `WRITE_BACK_EMBED`.

In [46]:
TABLES = {
    "factor_processes": factor_summary.reset_index(),      # the per-factor view, read this one
    "enrichr_terms": enrichr_results,
    "gprofiler_terms": gprofiler_results,
    "decoupler_regulators": decoupler_top,
}
# The gene-list size is part of the file name: TOP_N changes the ORA results (not decoupler's,
# which never sees the list), so two runs at different sizes have to sit side by side instead of
# one silently replacing the other.
print(f"config: score={SCORE_KEY}, cutoff={GENE_CUTOFF}, top_n={TOP_N}, "
      f"db={GSEAPY_DB}, network={DC_GENESET}")

for name, df in TABLES.items():
    if df is None or df.empty:
        print(f"[skip] {name} (empty)")
        continue
    path = TABLE_DIR / f"{name}_{RUN_ID}_top{TOP_N}.csv"
    df.to_csv(path, index=False)
    print(f"[write] {path} ({len(df):,} rows)")

# Into .uns, in the layout the tutorial's curation notebook expects.
for key, df in [("gseapy_enrichr_results", enrichr_results),
                ("gprofiler_results", gprofiler_results),
                ("decoupler_results", decoupler_top),
                ("factor_process_summary", factor_summary.reset_index())]:
    if df is not None and not df.empty:
        embed.uns[key] = df.convert_dtypes(convert_integer=False, convert_floating=False)

WRITE_BACK_EMBED = False
if WRITE_BACK_EMBED:
    ad.settings.allow_write_nullable_strings = True
    embed.write_h5ad(LATENT_H5AD)
    print(f"[write] {LATENT_H5AD}")

config: score=OOD_combined, cutoff=0.1, top_n=100, db=GO_Biological_Process_2023, network=collectri
[write] /home/albertoc/Desktop/QCB-Master-Thesis/05_drvi_tumoral_epi/tables/05_3_drvi_tum_32/factor_processes_drvi_tum_32_top100.csv (62 rows)
[write] /home/albertoc/Desktop/QCB-Master-Thesis/05_drvi_tumoral_epi/tables/05_3_drvi_tum_32/enrichr_terms_drvi_tum_32_top100.csv (1,226 rows)
[write] /home/albertoc/Desktop/QCB-Master-Thesis/05_drvi_tumoral_epi/tables/05_3_drvi_tum_32/gprofiler_terms_drvi_tum_32_top100.csv (164 rows)
[write] /home/albertoc/Desktop/QCB-Master-Thesis/05_drvi_tumoral_epi/tables/05_3_drvi_tum_32/decoupler_regulators_drvi_tum_32_top100.csv (22 rows)


### What this section adds, and what it does not

It adds a **second way of naming a factor**, and in this phase it is the primary one. SMI names
the factors that behave like a group - and the only groups available here are a clustering and a
borrowed label - while enrichment names the ones that behave like a programme, with a vocabulary
that owes nothing to CellTypist. Where both name the same factor they should be read together: a
factor that is both "leiden cluster 7" and "oxidative phosphorylation" is a tumour state with a
mechanism attached, which is more than either statement alone.

It does not add evidence. Three limits are structural, not tuning:

- **The vocabulary is 2,000 genes.** Any term whose genes are mostly outside the HVG panel cannot
  be found, and the ones that are found are found among the genes that vary most *between
  malignant cells*. This is a floor on coverage, and it is why the backgrounds are set
  explicitly.
- **ORA returns broad terms.** "Cell cycle", "translation", "response to interferon" is the
  resolution on offer; the specific branch of a pathway is not, and reading one into the term is
  the standard way to over-interpret this output. Finer readings come from the top genes against
  the literature.
- **A term is a hypothesis.** Convergence across the three tools raises confidence and does not
  replace validation. On this subset the cell cycle in particular will enrich somewhere, and a
  proliferation term is a fact about the axis, not a discovery.

`factor_processes_<run_id>_top<TOP_N>.csv` is the one artefact worth carrying forward: one row
per factor direction, its two SMI matches, its terms, its regulator and its top genes. It is the
notebook-level counterpart of what `05_7_factor_first` does against the collaborator's lists -
same scores, same background, a different question - and reading the two together is the point.

## Wrap-up

What this step leaves behind, and for whom.

`$DATA_DIR/05_tum/`:

- `model_<run_id>.pt` - the trained model. Only needed to recompute interpretability scores;
  nothing downstream loads it.
- `embed_<run_id>.h5ad` - the latent space with its per-dimension stats, its OOD/IND gene
  scores and (once the cells above have run) the SMI matches. **This is the file 05_4 reads**,
  and it needs neither the model nor a GPU.
- `shiao_tum_<run_id>.h5ad` - the 05_2 object with `obsm['X_drvi']`, for the steps that need
  genes and latent coordinates together (05_5, 05_6, 05_9).

`figures/05_3_<run_id>/` and `tables/05_3_<run_id>/` in the repo.

Two things to carry forward rather than leave implicit:

1. **The vanished count decides the size.** It is printed in the latent-dimensions section
   and in the log of `run_drvi_tum.py`. Nothing vanishing means 32 was too tight and the run
   is worth repeating at 64; the run id keeps the two apart, so it costs a training and no
   bookkeeping.
2. **No figure in this notebook groups cells by `cell_type`.** That column is `malignant`
   everywhere, by construction of the phase. Everything it would have grouped is grouped by
   `optscib_tum_leiden` first - the partition computed on these cells - and by
   `cell_type_01_4` second, a normal-breast CellTypist label used as a landmark and not as an
   identity. Whatever 05_4 onwards writes about "states" has to keep the same distinction, and
   the same order: the borrowed label is there to *name* a dimension the leiden partition and
   the gene programs have already established, never to establish one.

What this step does **not** do: the rare-state scan of 04_2 (*Finding rare (un-annotated) cell
types with DRVI*), which asks which dimensions describe a population the annotation does not
have. On this subset the annotation is one constant value, so "a sub-population the labels do
not resolve" would have to be defined against the leiden partition instead - a real question,
and a different one from the one 04_2 asks. It belongs with 05_6 (cell-first) rather than here.

The enrichment section below **is** included, and it is the counterpart of `05_7_factor_first`
at the level of "what is this axis about" rather than "does it enrich for the collaborator's
lists": same interpretability scores, same HVG background, public libraries instead of the
`.gmt`.